In [ ]:
import math
from collections import defaultdict, Counter
from sklearn.metrics import accuracy_score, classification_report


# ============================================================
# 1. FILE PATHS
# ============================================================

TRAIN_FILE = "en_ewt-ud-train.conllu"
DEV_FILE = "en_ewt-ud-dev.conllu"
TEST_FILE = "en_ewt-ud-test.conllu"


# ============================================================
# 2. LOAD CONLLU DATASET
# ============================================================

def load_conllu(filename):

    sentences = []
    words = []
    tags = []

    with open(filename, "r", encoding="utf-8") as file:

        for line in file:

            line = line.strip()

            # End of sentence
            if line == "":

                if words:
                    sentences.append((words, tags))
                    words = []
                    tags = []

                continue

            # Ignore comments
            if line.startswith("#"):
                continue

            columns = line.split("\t")

            # CoNLL-U has 10 columns
            if len(columns) != 10:
                continue

            token_id = columns[0]
            word = columns[1]
            pos_tag = columns[3]

            # Ignore multi-word tokens
            if "-" in token_id:
                continue

            # Ignore empty nodes
            if "." in token_id:
                continue

            words.append(word)
            tags.append(pos_tag)

    # Add final sentence
    if words:
        sentences.append((words, tags))

    return sentences


# ============================================================
# 3. LOAD TRAIN, DEV AND TEST DATA
# ============================================================

print("Loading UD English-EWT dataset...")

train_data = load_conllu(TRAIN_FILE)
dev_data = load_conllu(DEV_FILE)
test_data = load_conllu(TEST_FILE)

print("\nDataset loaded successfully!")

print("Training sentences   :", len(train_data))
print("Development sentences:", len(dev_data))
print("Test sentences       :", len(test_data))


# ============================================================
# 4. CALCULATE TRANSITION AND EMISSION COUNTS
# ============================================================

START = "<START>"
END = "<END>"

transition_counts = defaultdict(Counter)
emission_counts = defaultdict(Counter)
tag_counts = Counter()


for words, tags in train_data:

    previous_tag = START

    for word, tag in zip(words, tags):

        # Transition count
        transition_counts[previous_tag][tag] += 1

        # Emission count
        emission_counts[tag][word.lower()] += 1

        # POS tag count
        tag_counts[tag] += 1

        previous_tag = tag

    # Last tag -> END
    transition_counts[previous_tag][END] += 1


# Get all POS tags
tags = sorted(tag_counts.keys())

print("\nPOS Tags:")
print(tags)


# ============================================================
# 5. TRANSITION PROBABILITIES
# ============================================================

transition_prob = defaultdict(dict)

possible_next_tags = tags + [END]

for previous_tag in [START] + tags:

    total = sum(
        transition_counts[previous_tag].values()
    )

    for current_tag in possible_next_tags:

        count = transition_counts[
            previous_tag
        ][current_tag]

        # Laplace smoothing
        probability = (
            count + 1
        ) / (
            total + len(possible_next_tags)
        )

        transition_prob[
            previous_tag
        ][current_tag] = probability


# ============================================================
# 6. EMISSION PROBABILITIES
# ============================================================

emission_prob = defaultdict(dict)

for tag in tags:

    total = sum(
        emission_counts[tag].values()
    )

    vocabulary_size = len(
        emission_counts[tag]
    )

    for word, count in emission_counts[tag].items():

        # Laplace smoothing
        probability = (
            count + 1
        ) / (
            total + vocabulary_size + 1
        )

        emission_prob[tag][word] = probability


# ============================================================
# 7. UNKNOWN WORD HANDLING
# ============================================================

def get_emission_probability(word, tag):

    word = word.lower()

    if word in emission_prob[tag]:
        return emission_prob[tag][word]

    # Small probability for unknown words
    return 1 / (
        sum(emission_counts[tag].values()) + 100000
    )


# ============================================================
# 8. VITERBI ALGORITHM
# ============================================================

def viterbi(words):

    if not words:
        return []

    n = len(words)

    viterbi_table = [
        {}
        for _ in range(n)
    ]

    backpointer = [
        {}
        for _ in range(n)
    ]

    # --------------------------------------------------------
    # INITIALIZATION
    # --------------------------------------------------------

    first_word = words[0]

    for tag in tags:

        transition = transition_prob[
            START
        ][tag]

        emission = get_emission_probability(
            first_word,
            tag
        )

        viterbi_table[0][tag] = (
            math.log(transition)
            + math.log(emission)
        )

        backpointer[0][tag] = START


    # --------------------------------------------------------
    # RECURSION
    # --------------------------------------------------------

    for i in range(1, n):

        word = words[i]

        for current_tag in tags:

            emission = get_emission_probability(
                word,
                current_tag
            )

            best_score = -float("inf")
            best_previous_tag = None

            for previous_tag in tags:

                transition = transition_prob[
                    previous_tag
                ][current_tag]

                score = (
                    viterbi_table[i - 1][previous_tag]
                    + math.log(transition)
                    + math.log(emission)
                )

                if score > best_score:

                    best_score = score
                    best_previous_tag = previous_tag

            viterbi_table[i][current_tag] = best_score

            backpointer[i][current_tag] = (
                best_previous_tag
            )


    # --------------------------------------------------------
    # TERMINATION
    # --------------------------------------------------------

    best_last_tag = max(
        viterbi_table[-1],
        key=lambda tag:
        viterbi_table[-1][tag]
        + math.log(
            transition_prob[tag][END]
        )
    )


    # --------------------------------------------------------
    # BACKTRACKING
    # --------------------------------------------------------

    best_tags = [best_last_tag]

    for i in range(n - 1, 0, -1):

        best_tags.append(
            backpointer[i][
                best_tags[-1]
            ]
        )

    best_tags.reverse()

    return best_tags


# ============================================================
# 9. ACCEPT SENTENCE AS INPUT
# ============================================================

print("\n" + "=" * 60)
print("HMM POS TAGGER")
print("=" * 60)

sentence = input(
    "\nEnter a sentence: "
)

words = sentence.split()


# ============================================================
# 10. PREDICT POS TAGS
# ============================================================

predicted_tags = viterbi(words)

print("\nPredicted POS Tags")
print("-" * 40)

for word, tag in zip(
    words,
    predicted_tags
):

    print(
        f"{word} -> {tag}"
    )


# ============================================================
# 11. COMPARE WITH ACTUAL TAGS
# ============================================================

actual_input = input(
    "\nEnter actual POS tags "
    "(or press Enter to skip): "
)

if actual_input.strip():

    actual_tags = actual_input.split()

    if len(actual_tags) == len(predicted_tags):

        print("\nComparison")
        print("-" * 60)

        for word, actual, predicted in zip(
            words,
            actual_tags,
            predicted_tags
        ):

            print(
                f"{word:15}"
                f"Actual: {actual:8}"
                f"Predicted: {predicted}"
            )

        sentence_accuracy = accuracy_score(
            actual_tags,
            predicted_tags
        )

        print(
            f"\nSentence Accuracy: "
            f"{sentence_accuracy * 100:.2f}%"
        )

    else:

        print(
            "\nNumber of actual tags "
            "does not match number of words."
        )


# ============================================================
# 12. EVALUATE ON TEST DATASET
# ============================================================

print("\n" + "=" * 60)
print("EVALUATING ON TEST DATASET")
print("=" * 60)

print("\nPlease wait...")

y_true = []
y_pred = []

for i, (words, actual_tags) in enumerate(test_data):

    predicted_tags = viterbi(words)

    y_true.extend(actual_tags)
    y_pred.extend(predicted_tags)

    if (i + 1) % 100 == 0:

        print(
            f"Processed {i + 1}/"
            f"{len(test_data)} sentences"
        )


# ============================================================
# 13. ACCURACY
# ============================================================

accuracy = accuracy_score(
    y_true,
    y_pred
)

print("\n" + "=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

print(
    f"\nPOS Tagging Accuracy: "
    f"{accuracy * 100:.2f}%"
)


# ============================================================
# 14. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report")
print("-" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        zero_division=0
    )
)


# ============================================================
# 15. SUMMARY
# ============================================================

print("=" * 60)
print("PROJECT SUMMARY")
print("=" * 60)

print("\nDataset       : UD English-EWT")
print("Model         : Hidden Markov Model (HMM)")
print("Algorithm     : Viterbi Algorithm")
print(
    "Training data :",
    len(train_data),
    "sentences"
)
print(
    "Test data     :",
    len(test_data),
    "sentences"
)
print(
    f"Test Accuracy : "
    f"{accuracy * 100:.2f}%"
)

Loading UD English-EWT dataset...

Dataset loaded successfully!
Training sentences   : 12544
Development sentences: 2001
Test sentences       : 2077

POS Tags:
['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']

HMM POS TAGGER



Enter a sentence:  The student reads a book.



Predicted POS Tags
----------------------------------------
The -> DET
student -> NOUN
reads -> VERB
a -> DET
book. -> NOUN



Enter actual POS tags (or press Enter to skip):  



EVALUATING ON TEST DATASET

Please wait...
Processed 100/2077 sentences
Processed 200/2077 sentences
Processed 300/2077 sentences
Processed 400/2077 sentences
Processed 500/2077 sentences
Processed 600/2077 sentences
Processed 700/2077 sentences
Processed 800/2077 sentences
